# Original

In [6]:
# 2. Importar todas las dependencias necesarias
from copy import deepcopy
import cobra
from cobra.io import load_model
from optlang import gurobi_interface
from cobra import Reaction, Model, Metabolite # Importación añadida
from optlang.gurobi_interface import Model as GurobiModel, Variable, Constraint, Objective
from optlang.symbolics import add
import numpy as np
import cobra
from cobra.io import load_model
import pandas as pd
import gurobipy
import time
from cobra.util import create_stoichiometric_matrix 

In [2]:
#Importar el modelo
model = load_model("iJO1366")

#Establecer los nutrientes en cero
model.reactions.EX_glc__D_e.lower_bound = 0 
model.reactions.EX_o2_e.lower_bound = 0    
#establecer la reacción objetivo como ATPM y optimizar  
model.objective = "ATPM"  
solution = model.optimize()
print(f'La produción de ATPM es:{solution.objective_value}')

Set parameter Username
Set parameter LicenseID to value 2608033
Academic license - for non-commercial use only - expires 2026-01-09
La produción de ATPM es:None


c:\Users\diazt\anaconda3\lib\site-packages\cobra\util\solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


Vemos que el modelo no tiene EGCS, activamos las reacciones que producen los EGCs

In [3]:
#Hacemos una copia del modelo
model_test=deepcopy(model)

#Activamos las reacciones
egc_rxn_ids = [ "SPODM", "SPODMpp", "SUCASPtpp", "SUCFUMtpp", "SUCMALtpp", "SUCTARTtpp"]

# Reactivamos
for rxn_id in egc_rxn_ids:
    try:
        rxn = model_test.reactions.get_by_id(rxn_id)
        rxn.lower_bound = -1000  # reversible (si tiene sentido) o permite flujo
        rxn.upper_bound = 1000
        print(f"Reactivada: {rxn.id} ({rxn.name})")
    except KeyError:
        print(f"Reacción {rxn_id} no encontrada en el modelo")

# Optimizamos la reacción de disipación de ATP
solution_test= model_test.optimize()
print(f"Flujo de disipación de ATP con reacciones reactivadas: {solution_test.objective_value}")

Read LP format model from file C:\Users\diazt\AppData\Local\Temp\tmphe7q1bm9.lp
Reading time = 0.26 seconds
: 1805 rows, 5166 columns, 20366 nonzeros
Reactivada: SPODM (Superoxide dismutase)
Reactivada: SPODMpp (Superoxide dismutase)
Reactivada: SUCASPtpp (Succinate:aspartate antiporter (periplasm))
Reactivada: SUCFUMtpp (Succinate:fumarate antiporter (periplasm))
Reactivada: SUCMALtpp (Succinate:malate antiporter (periplasm))
Reactivada: SUCTARTtpp (Succinate:D-tartrate antiporter (periplasm))
Flujo de disipación de ATP con reacciones reactivadas: 750.0


En efecto ahora hay EGCS, reactivamos los nutrientes y empezamos a preparar el modelo para la optimización

In [4]:
#Activamos los nutrientes
model_test.reactions.EX_glc__D_e.lower_bound = -1000  # Cortar glucosa
model_test.reactions.EX_o2_e.lower_bound = -1000  
#Optimizamos para ver la producción de ATPM
solution_test= model_test.optimize()
print(f"Flujo de disipación de ATP con reacciones y nutrientes reactivados: {solution_test.objective_value}")

Flujo de disipación de ATP con reacciones y nutrientes reactivados: 1000.0


In [5]:
#Dejamos el modelo solo con reacciones reversibles
def prepare_model_for_globalfit(model, biomass_rxn_id="BIOMASS_Ec_iJO1366_core_53p95M", atp_rxn_id="ATPM"):
    model_irrev = deepcopy(model)

    # 2. Identificar reacciones reversibles (excluyendo biomasa y ATPM)
    reversible_rxns = [
        rxn for rxn in model_irrev.reactions 
        if (rxn.lower_bound < 0) and 
           (rxn.id != biomass_rxn_id) and 
           (rxn.id != atp_rxn_id)
    ]

    # 3. Crear reacciones inversas (excluyendo biomasa y ATPM)
    reverse_reactions = []
    for rxn in reversible_rxns:
        rxn_reverse = rxn.copy()
        rxn_reverse.id = f"reverse_{rxn.id}"
        rxn_reverse.lower_bound = 0
        rxn_reverse.upper_bound = abs(rxn.lower_bound)
        reverse_reactions.append(rxn_reverse)

    # 4. Añadir reacciones inversas al modelo
    if reverse_reactions:
        model_irrev.add_reactions(reverse_reactions)

    # 5. Hacer las originales irreversibles (excepto biomasa y ATPM)
    for rxn in reversible_rxns:
        rxn.lower_bound = 0

    return model_irrev
   

In [23]:
#Obtenemos la matriz estequiometrica, con la matriz S_g en la esquina superior izquierda y S_ng en la inferior derecha (el resto ceros)
def get_extended_stoichiometry_matrix(model):
    """Generates extended stoichiometric matrix with distinct indices"""
    original_mets = [m.id for m in model.metabolites]
    ng_mets = [f"{m.id}_ng" for m in model.metabolites]
    n_mets = len(original_mets)
    
    original_rxns = [rxn.id for rxn in model.reactions]
    ng_rxns = [f"ng_{rxn.id}" for rxn in model.reactions]
    
    S_original = create_stoichiometric_matrix(model, array_type='dense')
    S_ng = S_original.copy()
    
    top = np.hstack([S_original, np.zeros((n_mets, len(ng_rxns)))])
    bottom = np.hstack([np.zeros((n_mets, len(original_rxns))), S_ng])
    S_extended = np.vstack([top, bottom])
    
    extended_mets = original_mets + ng_mets
    extended_rxns = original_rxns + ng_rxns
    
    return pd.DataFrame(S_extended, index=extended_mets, columns=extended_rxns)


In [19]:
#Implementamos Globalfit, dejamos establecida una lista mas pequeña para ver si funciona
def globalfit_with_print(model, S_g_ng_original, biomass_rxn_id, atp_rxn_id, T_g=0.0001, extra_exclude=[]):
    """Corrected GlobalFit implementation"""
    print("\n=== STARTING GLOBALFIT ===")
    
    # 1. Configure Gurobi solver
    opt_model = gurobi_interface.Model(name="GLOBALFIT_GUROBI")
    opt_model.problem.Params.Presolve = 1
    opt_model.problem.Params.Threads = 4
    
    # 2. Filter relevant reactions
    rxns_to_use = [rxn.id for rxn in model.reactions 
                  if not (rxn.id.startswith(("EX_", "DM_", "SK_"))) 
                  and not rxn.boundary 
                  and "tex" not in rxn.id 
                  and len(rxn.compartments) <= 1
                  and rxn.id not in extra_exclude]+['BIOMASS_Ec_iJO1366_core_53p95M']
    rxns_to_use=['SPODM', 'SPODMpp', 'SUCASPtpp', 'SUCFUMtpp', 'SUCMALtpp', 'SUCTARTtpp','UPP3MT', 'URACPAH', 'XTSNH', 'Zn2tex','BIOMASS_Ec_iJO1366_core_53p95M','ATPM']
    rxns_g = rxns_to_use
    rxns_ng = [f"ng_{r}" for r in rxns_to_use]
    selected_rxns = rxns_g + rxns_ng
    
    # 3. Filter stoichiometric matrix (CORRECTED)
    S_g_ng = S_g_ng_original[selected_rxns].copy()
    S_g_ng = S_g_ng.loc[(S_g_ng != 0).any(axis=1)]
    
    # 4. Variables
    v_g = {rxn: gurobi_interface.Variable(f"v_g_{rxn}", lb=-1000, ub=1000) for rxn in rxns_g}
    v_ng = {rxn: gurobi_interface.Variable(f"v_ng_{rxn}", lb=-1000, ub=1000) for rxn in rxns_ng}
    delta = {rxn: gurobi_interface.Variable(f"delta_{rxn}", type="binary") for rxn in rxns_g}
    lambda_dual = {met: gurobi_interface.Variable(f"lambda_{met}", lb=-1000, ub=1000) for met in S_g_ng.index[len(model.metabolites):]}
    mu = {rxn: gurobi_interface.Variable(f"mu_{rxn}", lb=0) for rxn in rxns_ng}

    # 5. Constraints (FULLY CORRECTED)
    print("\nAdding constraints...")
    
    for rxn in model.reactions:
        if rxn.id.startswith('EX_'):
            if f"ng_{rxn.id}" in v_ng:
                opt_model.add(gurobi_interface.Constraint(v_ng[f"ng_{rxn.id}"], lb=0, ub=0, name=f"no_uptake_{rxn.id}"))

    # Mass balance for growth (CORRECTED)
    for met in S_g_ng.index[:len(model.metabolites)]:
        expr = 0
        for rxn in rxns_g:
            coeff = S_g_ng.at[met, rxn]  # Direct scalar access
            if isinstance(coeff, (int, float)) and not np.isclose(coeff, 0, atol=1e-8):
                expr += float(coeff) * v_g[rxn]
        if expr != 0:  # Only add non-zero constraints
            opt_model.add(gurobi_interface.Constraint(expr, lb=0, ub=0, name=f"growth_{met}"))
    
    # Minimum biomass
    opt_model.add(gurobi_interface.Constraint(
        v_g[biomass_rxn_id], lb=T_g, name="min_biomass"))
    
    # Flux bounds
    for rxn in rxns_g:
        ub1 = model.reactions.get_by_id(rxn).upper_bound
        opt_model.add(gurobi_interface.Constraint(
            v_g[rxn] - ub1 * (1 - delta[rxn]), ub=0, name=f"flux_ub_{rxn}"))
        
    # Balance de masa sin crecimiento
    for met in S_g_ng.index[len(model.metabolites):]:
        expr = sum(S_g_ng.at[met, rxn] * v_ng[rxn] for rxn in rxns_ng if not np.isclose(S_g_ng.at[met, rxn], 0))
        opt_model.add(gurobi_interface.Constraint(expr, lb=0, ub=0, name=f"mass_balance_nogrowth_{met}"))

    # Cotas v_min y v_max ajustadas por delta
    for rxn in rxns_ng:
        rxn_orig = rxn.replace("ng_", "")
        ub2 = model.reactions.get_by_id(rxn_orig).upper_bound
        opt_model.add(gurobi_interface.Constraint(v_ng[rxn] - ub2 * (1 - delta[rxn_orig]), ub=0, name=f"flux_bound_ng_upper_{rxn}"))
    
    # Gradiente Lagrangiano: c + lambda*S + mu = 0
    # c = 1 solo en ng_ATP
    for rxn in rxns_ng:
        grad_expr = 0
        for met in lambda_dual:
            coeff = S_g_ng.at[met, rxn]
            if not np.isclose(coeff, 0):
                grad_expr += coeff * lambda_dual[met]
        if rxn == f"ng_{atp_rxn_id}":
            grad_expr += 1  # c_ATP = 1
        grad_expr += mu[rxn]
        opt_model.add(gurobi_interface.Constraint(grad_expr, ub=0, lb=0, name=f"lagrangian_{rxn}"))

    print("  - Condiciones de complementariedad...", end=" ")
    M = 1e6
    count = 0
    for rxn in rxns_g:
        opt_model.add(gurobi_interface.Constraint(
            expression=mu['ng_'+rxn] - M * (1-delta[rxn]),
            ub=0,
            name=f"complementarity_{rxn}"
        ))
        count += 1
    print(f"{count} restricciones agregadas")
    
    #complementariedad 2
    expr_sum_mu = sum(mu.values())
    opt_model.add(gurobi_interface.Constraint(expr_sum_mu - v_ng[f"ng_{atp_rxn_id}"], lb=0, ub=0, name="atp_dual_match"))

    # 8. v_ATP ≈ 0 en no-crecimiento
    if f"ng_{atp_rxn_id}" in rxns_ng:
        print("  - Restricción de ATP en no-crecimiento...", end=" ")
        opt_model.add(gurobi_interface.Constraint(
            expression=v_ng[f"ng_{atp_rxn_id}"],
            lb=0,
            ub=1e-6,
            name="zero_atp_maintenance"
        ))
        print("OK")
        
    # 6. Solve
    print("\Establecer Objetivo")
    opt_model.objective = gurobi_interface.Objective(sum(delta.values()), direction="min")
    return opt_model

In [13]:
def prepare_model_for_globalfit(model, biomass_rxn_id="BIOMASS_Ec_iJO1366_core_53p95M", atp_rxn_id="ATPM"):
    model_irrev = deepcopy(model)
    
    # 1. Configurar objetivo solo a biomasa (asegurando coeficiente correcto)
    model_irrev.objective = biomass_rxn_id
    for rxn in model_irrev.reactions:
        rxn.objective_coefficient = 1.0 if rxn.id == biomass_rxn_id else 0.0
    
    # 2. Identificar reacciones reversibles (EXCLUYENDO biomasa explícitamente)
    reversible_rxns = [
        rxn for rxn in model_irrev.reactions 
        if (rxn.lower_bound < 0) and 
           (rxn.id != biomass_rxn_id) and 
           (rxn.id != atp_rxn_id) and
           (not rxn.id.endswith("_reverse"))  # Excluir cualquier reversa existente
    ]

    # 3. Eliminar cualquier versión reversa de biomasa si existe
    biomass_reverse_ids = [rxn.id for rxn in model_irrev.reactions 
                         if rxn.id.startswith(f"{biomass_rxn_id}_reverse")]
    if biomass_reverse_ids:
        model_irrev.remove_reactions(biomass_reverse_ids)
    
    # [Resto del código igual...]
    
    return model_irrev

In [24]:
#Dejamos el modelo irreversible
model_irrev = prepare_model_for_globalfit(
    model_test,
    biomass_rxn_id="BIOMASS_Ec_iJO1366_core_53p95M",  # Tu reacción de biomasa
    atp_rxn_id="ATPM"                                 # Tu reacción de ATP
)
#Generamos la matriz estequiométrica extendida
S_g_ng = get_extended_stoichiometry_matrix(model_irrev)

Read LP format model from file C:\Users\diazt\AppData\Local\Temp\tmppkc0dxon.lp
Reading time = 0.07 seconds
: 1805 rows, 5166 columns, 20366 nonzeros


In [27]:
print(model_irrev.optimize().objective_value)

None


c:\Users\diazt\anaconda3\lib\site-packages\cobra\util\solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


c:\Users\diazt\anaconda3\lib\site-packages\cobra\util\solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


None


In [12]:
model_irrev

Name,iJO1366
Memory address,15f161af880
Number of metabolites,1805
Number of reactions,3225
Number of genes,1367
Number of groups,37
Objective expression,1.0*BIOMASS_Ec_iJO1366_core_53p95M - 1.0*BIOMASS_Ec_iJO1366_core_53p95M_reverse_5c8b1
Compartments,"cytosol, extracellular space, periplasm"


In [9]:
# Ejecutar
opt_model = globalfit_with_print(
    model=model_irrev,
    S_g_ng_original=S_g_ng,
    biomass_rxn_id="BIOMASS_Ec_iJO1366_core_53p95M",
    atp_rxn_id="ATPM",
    T_g=0.0001, 
    extra_exclude=[]
)


=== STARTING GLOBALFIT ===

Adding constraints...
  - Condiciones de complementariedad... 12 restricciones agregadas
  - Restricción de ATP en no-crecimiento... OK

Solving with Gurobi...


No logra obtener la solución en ese espacio menos de soluciones!!

In [10]:
opt_model.optimize()

'infeasible'

# Intentar solucionar model irrev

In [7]:
from cobra import Model, Reaction, Metabolite
from cobra.io import load_model, read_sbml_model
import cobra
import pandas as pd
import copy
from copy import deepcopy

# 1. Cargar el modelo fresco
model = load_model("iJO1366")


# 3. Configurar objetivo de biomasa
model.objective = "ATPM"

# 4. Verificar solución básica
solution = model.optimize()
print(f"Flujo de biomasa: {solution.objective_value}")

Flujo de biomasa: 234.9999999999999


In [8]:
model.optimize().objective_value

234.9999999999999

In [9]:
model_test0=deepcopy(model)
#Establecer los nutrientes en cero
model_test0.reactions.EX_glc__D_e.lower_bound = 0 
model_test0.reactions.EX_o2_e.lower_bound = 0    
#establecer la reacción objetivo como ATPM y optimizar  
model_test0.objective = "ATPM"  
solution = model_test0.optimize()
print(f'La produción de ATPM es:{solution.objective_value}')

Read LP format model from file C:\Users\diazt\AppData\Local\Temp\tmptfb6y5kk.lp
Reading time = 0.09 seconds
: 1805 rows, 5166 columns, 20366 nonzeros
La produción de ATPM es:None


model_test.objective="ATPM"
model_test.optimize().objective_value

In [10]:
def prepare_model_for_globalfit_fixed(model, biomass_rxn_id="BIOMASS_Ec_iJO1366_core_53p95M", atp_rxn_id="ATPM"):
    model_irrev = deepcopy(model)
    model_irrev.objective = atp_rxn_id

    reversible_rxns = [
        rxn for rxn in model_irrev.reactions
        if rxn.lower_bound < 0
        and not rxn.id.startswith("EX_")]

    for rxn in reversible_rxns:
        rxn_reverse = rxn.copy()
        rxn_reverse.id = f"{rxn.id}_reversed"
        rxn_reverse.lower_bound = 0
        rxn_reverse.upper_bound = abs(rxn.lower_bound)  # importante: lower_bound es negativo
        for met, coeff in rxn_reverse.metabolites.items():
            rxn_reverse.add_metabolites({met: -2 * coeff})  # invertir estequiometría

        rxn.lower_bound = 0
        model_irrev.add_reactions([rxn_reverse])

    return model_irrev


In [11]:
# 1. Preparar modelo
model_irrev = prepare_model_for_globalfit_fixed(model, biomass_rxn_id="BIOMASS_Ec_iJO1366_core_53p95M", atp_rxn_id="ATPM")

# 2. Verificar reacciones
print("\nReacciones de biomasa:")
print([rxn.id for rxn in model_irrev.reactions if "BIOMASS" in rxn.id])

# 3. Verificar objetivo
print("\nExpresión objetivo:")
print(model_irrev.objective.expression)

# 4. Optimizar
solution = model_irrev.optimize()
print(f"\nFlujo de ATPM: {solution.objective_value}")



Read LP format model from file C:\Users\diazt\AppData\Local\Temp\tmpkun9_pcp.lp
Reading time = 0.10 seconds
: 1805 rows, 5166 columns, 20366 nonzeros

Reacciones de biomasa:
['BIOMASS_Ec_iJO1366_WT_53p95M', 'BIOMASS_Ec_iJO1366_core_53p95M']

Expresión objetivo:
1.0*ATPM - 1.0*ATPM_reverse_5b752

Flujo de ATPM: 234.9999999999999


In [12]:
#Establecer los nutrientes en cero
model_irrev.reactions.EX_glc__D_e.lower_bound = -1000 
model_irrev.reactions.EX_o2_e.lower_bound = -1000   
#establecer la reacción objetivo como ATPM y optimizar  
model_irrev.objective = "ATPM"  
solution = model_irrev.optimize()
print(f'La produción de ATPM es:{solution.objective_value}')

La produción de ATPM es:1000.0


In [13]:
model_irrev.reactions.EX_o2_e.id

'EX_o2_e'

In [14]:
#Hacemos una copia del modelo
model_test=deepcopy(model_irrev)

#Activamos las reacciones
egc_rxn_ids = [ "SPODM", "SPODMpp", "SUCASPtpp", "SUCFUMtpp", "SUCMALtpp", "SUCTARTtpp"]

# Reactivamos
for rxn_id in egc_rxn_ids:
    try:
        rxn = model_test.reactions.get_by_id(rxn_id)
        rxn.lower_bound = -1000  # reversible (si tiene sentido) o permite flujo
        rxn.upper_bound = 1000
        print(f"Reactivada: {rxn.id} ({rxn.name})")
    except KeyError:
        print(f"Reacción {rxn_id} no encontrada en el modelo")

# Optimizamos la reacción de disipación de ATP
solution_test= model_test.optimize()
print(f"Flujo de disipación de ATP con reacciones reactivadas: {solution_test.objective_value}")

Read LP format model from file C:\Users\diazt\AppData\Local\Temp\tmpwqzpnbli.lp
Reading time = 0.09 seconds
: 1805 rows, 6388 columns, 23968 nonzeros
Reactivada: SPODM (Superoxide dismutase)
Reactivada: SPODMpp (Superoxide dismutase)
Reactivada: SUCASPtpp (Succinate:aspartate antiporter (periplasm))
Reactivada: SUCFUMtpp (Succinate:fumarate antiporter (periplasm))
Reactivada: SUCMALtpp (Succinate:malate antiporter (periplasm))
Reactivada: SUCTARTtpp (Succinate:D-tartrate antiporter (periplasm))
Flujo de disipación de ATP con reacciones reactivadas: 1000.0


In [15]:
#Obtenemos la matriz estequiometrica, con la matriz S_g en la esquina superior izquierda y S_ng en la inferior derecha (el resto ceros)
def get_extended_stoichiometry_matrix(model):
    """Generates extended stoichiometric matrix with distinct indices"""
    original_mets = [m.id for m in model.metabolites]
    ng_mets = [f"{m.id}_ng" for m in model.metabolites]
    n_mets = len(original_mets)
    
    original_rxns = [rxn.id for rxn in model.reactions]
    ng_rxns = [f"ng_{rxn.id}" for rxn in model.reactions]
    
    S_original = create_stoichiometric_matrix(model, array_type='dense')
    S_ng = S_original.copy()
    
    top = np.hstack([S_original, np.zeros((n_mets, len(ng_rxns)))])
    bottom = np.hstack([np.zeros((n_mets, len(original_rxns))), S_ng])
    S_extended = np.vstack([top, bottom])
    
    extended_mets = original_mets + ng_mets
    extended_rxns = original_rxns + ng_rxns
    
    return pd.DataFrame(S_extended, index=extended_mets, columns=extended_rxns)

In [16]:
from cobra.util import create_stoichiometric_matrix 
import numpy as np
S_g_ng=get_extended_stoichiometry_matrix(model_test)

In [17]:
#Implementamos Globalfit, dejamos establecida una lista mas pequeña para ver si funciona
def globalfit_with_print(model, S_g_ng_original, biomass_rxn_id, atp_rxn_id, T_g=0.0001, extra_exclude=[]):
    """Corrected GlobalFit implementation"""
    print("\n=== STARTING GLOBALFIT ===")
    
    # 1. Configure Gurobi solver
    opt_model = gurobi_interface.Model(name="GLOBALFIT_GUROBI")
    opt_model.problem.Params.Presolve = 1
    opt_model.problem.Params.Threads = 4
    
    # 2. Filter relevant reactions
    rxns_to_use = [rxn.id for rxn in model.reactions 
                  if not (rxn.id.startswith(("EX_", "DM_", "SK_"))) 
                  and not rxn.boundary 
                  and "tex" not in rxn.id 
                  and len(rxn.compartments) <= 1
                  and rxn.id not in extra_exclude]+['BIOMASS_Ec_iJO1366_core_53p95M']
    rxns_to_use=['SPODM', 'SPODMpp', 'SUCASPtpp', 'SUCFUMtpp', 'SUCMALtpp', 'SUCTARTtpp','UPP3MT', 'URACPAH', 'XTSNH', 'Zn2tex','BIOMASS_Ec_iJO1366_core_53p95M','ATPM']
    rxns_g = rxns_to_use
    rxns_ng = [f"ng_{r}" for r in rxns_to_use]
    selected_rxns = rxns_g + rxns_ng
    
    # 3. Filter stoichiometric matrix (CORRECTED)
    S_g_ng = S_g_ng_original[selected_rxns].copy()
    S_g_ng = S_g_ng.loc[(S_g_ng != 0).any(axis=1)]
    
    # 4. Variables
    v_g = {rxn: gurobi_interface.Variable(f"v_g_{rxn}", lb=-1000, ub=1000) for rxn in rxns_g}
    v_ng = {rxn: gurobi_interface.Variable(f"v_ng_{rxn}", lb=-1000, ub=1000) for rxn in rxns_ng}
    delta = {rxn: gurobi_interface.Variable(f"delta_{rxn}", type="binary") for rxn in rxns_g}
    lambda_dual = {met: gurobi_interface.Variable(f"lambda_{met}", lb=-1000, ub=1000) for met in S_g_ng.index[len(model.metabolites):]}
    mu = {rxn: gurobi_interface.Variable(f"mu_{rxn}", lb=0) for rxn in rxns_ng}
    z= {rxn: gurobi_interface.Variable(f"z_{rxn}", type="binary") for rxn in rxns_ng}
    # 5. Constraints (FULLY CORRECTED)
    print("\nAdding constraints...")
    
    #v_ng sin nutrientes
    for rxn in model.reactions:
        if rxn.id.startswith('EX_'):
            if f"ng_{rxn.id}" in v_ng:
                opt_model.add(gurobi_interface.Constraint(v_ng[f"ng_{rxn.id}"], lb=0, ub=0, name=f"no_uptake_{rxn.id}"))

    # balance de masa por crecimiento
    for met in S_g_ng.index[:len(model.metabolites)]:
        expr = 0
        for rxn in rxns_g:
            coeff = S_g_ng.at[met, rxn]  # Direct scalar access
            if isinstance(coeff, (int, float)) and not np.isclose(coeff, 0, atol=1e-8):
                expr += float(coeff) * v_g[rxn]
        if expr != 0:  # Only add non-zero constraints
            opt_model.add(gurobi_interface.Constraint(expr, lb=0, ub=0, name=f"growth_{met}"))
    
    # Minimum biomass
    opt_model.add(gurobi_interface.Constraint(
        v_g[biomass_rxn_id], lb=T_g, name="min_biomass"))
    
    # Flux bounds
    for rxn in rxns_g:
        ub1 = model.reactions.get_by_id(rxn).upper_bound
        opt_model.add(gurobi_interface.Constraint(
            v_g[rxn] - ub1 * (1 - delta[rxn]), ub=0, name=f"flux_ub_{rxn}"))
        
    # Balance de masa sin crecimiento
    for met in S_g_ng.index[len(model.metabolites):]:
        expr = sum(S_g_ng.at[met, rxn] * v_ng[rxn] for rxn in rxns_ng if not np.isclose(S_g_ng.at[met, rxn], 0))
        opt_model.add(gurobi_interface.Constraint(expr, lb=0, ub=0, name=f"mass_balance_nogrowth_{met}"))

    # Cotas v_min y v_max ajustadas por delta
    for rxn in rxns_ng:
        rxn_orig = rxn.replace("ng_", "")
        ub2 = model.reactions.get_by_id(rxn_orig).upper_bound
        opt_model.add(gurobi_interface.Constraint(v_ng[rxn] - ub2 * (1 - delta[rxn_orig]), ub=0, name=f"flux_bound_ng_upper_{rxn}"))
    
    # Gradiente Lagrangiano: c + lambda*S + mu = 0
    # c = 1 solo en ng_ATP
    for rxn in rxns_ng:
        grad_expr = 0
        for met in lambda_dual:
            coeff = S_g_ng.at[met, rxn]
            if not np.isclose(coeff, 0):
                grad_expr += coeff * lambda_dual[met]
        if rxn == f"ng_{atp_rxn_id}":
            grad_expr += -1  # c_ATP = 1
        grad_expr += mu[rxn]
        opt_model.add(gurobi_interface.Constraint(grad_expr, ub=0, lb=0, name=f"lagrangian_{rxn}"))

    print("  - Condiciones de complementariedad...", end=" ")
    M = 1e6
    count = 0
    for rxn in rxns_ng:
        opt_model.add(gurobi_interface.Constraint(
            expression=mu[rxn] - M * (z[rxn]),
            lb=0,
            name=f"complementarity_{rxn}"
        ))
        count += 1
    print(f"{count} restricciones agregadas")
    
    #complementariedad 2
    for rxn in rxns_ng:
        rxn_orig = rxn.replace("ng_", "")
        ub2 = model.reactions.get_by_id(rxn_orig).upper_bound
        opt_model.add(gurobi_interface.Constraint(-v_ng[rxn] + ub2 * (1 - delta[rxn_orig])-M*(1-z[rxn]), ub=0, name=f"comp_{rxn}"))

    # 8. v_ATP ≈ 0 en no-crecimiento
    if f"ng_{atp_rxn_id}" in rxns_ng:
        print("  - Restricción de ATP en no-crecimiento...", end=" ")
        opt_model.add(gurobi_interface.Constraint(
            expression=v_ng[f"ng_{atp_rxn_id}"],
            lb=0,
            ub=1e-6,
            name="zero_atp_maintenance"
        ))
        print("OK")
        
    # 6. Solve
    print("\nSolving with Gurobi...")
    opt_model.objective = gurobi_interface.Objective(sum(delta.values()), direction="min")
    return opt_model

In [18]:
def prepare_model_for_globalfit_fixed(model, biomass_rxn_id="BIOMASS_Ec_iJO1366_core_53p95M", atp_rxn_id="ATPM"):
    # 1. Hacer una copia limpia del modelo
    model_irrev = deepcopy(model)  # Cambiado a copy() para mejor compatibilidad
    
    # 2. Configurar el objetivo PRIMERO
    model_irrev.objective = biomass_rxn_id
    
    # 4. Eliminar CUALQUIER reacción inversa existente
    reverse_rxns = [rxn.id for rxn in model_irrev.reactions 
                   if "_reverse" in rxn.id or "_reverse_" in rxn.id]
    if reverse_rxns:
        model_irrev.remove_reactions(reverse_rxns)
       # Aporte de oxígeno
    
    # 6. Identificar reacciones reversibles (excluyendo especiales)
    reversible_rxns = [
        rxn for rxn in model_irrev.reactions 
        if rxn.lower_bound < 0 
        and rxn.id not in [biomass_rxn_id, atp_rxn_id]
    ]
    
    # 7. Crear reacciones inversas solo para las necesarias
    for rxn in reversible_rxns:
        rxn_reverse = rxn.copy()
        rxn_reverse.id = f"{rxn.id}_reversed"
        rxn_reverse.lower_bound = 0
        rxn_reverse.upper_bound = abs(rxn.lower_bound)
        model_irrev.add_reactions([rxn_reverse])
        rxn.lower_bound = 0  # Hacer la original irreversible
    
    # 8. Verificación final del objetivo
    model_irrev.objective = biomass_rxn_id
    for rxn in model_irrev.reactions:
        rxn.objective_coefficient = 1.0 if rxn.id == biomass_rxn_id else 0.0
    
    return model_irrev

In [19]:
modelo_gf=globalfit_with_print(model=model_test, S_g_ng_original=S_g_ng, biomass_rxn_id="BIOMASS_Ec_iJO1366_core_53p95M", atp_rxn_id="ATPM", T_g=0.0001, extra_exclude=[])


=== STARTING GLOBALFIT ===

Adding constraints...
  - Condiciones de complementariedad... 12 restricciones agregadas
  - Restricción de ATP en no-crecimiento... OK

Solving with Gurobi...


In [20]:
modelo_gf.optimize()

'infeasible'

In [21]:
def globalfit_corrected(model, S_g_ng_original, biomass_rxn_id, atp_rxn_id, T_g=0.0001, extra_exclude=[]):
    print("\n=== STARTING GLOBALFIT (CORRECTED) ===")
    
    # 1. Configuración del solver
    opt_model = gurobi_interface.Model(name="GLOBALFIT_GUROBI_CORRECTED")
    opt_model.problem.Params.Presolve = 1
    opt_model.problem.Params.Threads = 4
    
    # 2. Selección de reacciones (simplificada para prueba)
    rxns_g = ['SPODM', 'SPODMpp', 'SUCASPtpp', 'SUCFUMtpp', 'SUCMALtpp', 'SUCTARTtpp',
              'UPP3MT', 'URACPAH', 'XTSNH', 'Zn2tex', 'BIOMASS_Ec_iJO1366_core_53p95M', 'ATPM']
    rxns_ng = [f"ng_{r}" for r in rxns_g]
    
    # 3. Filtrar matriz estequiométrica
    S_g_ng = S_g_ng_original[rxns_g + rxns_ng].copy()
    S_g_ng = S_g_ng.loc[(S_g_ng != 0).any(axis=1)]
    
    # 4. Variables
    v_g = {rxn: gurobi_interface.Variable(f"v_g_{rxn}", lb=-1000, ub=1000) for rxn in rxns_g}
    v_ng = {rxn: gurobi_interface.Variable(f"v_ng_{rxn}", lb=-1000, ub=1000) for rxn in rxns_ng}
    delta = {rxn: gurobi_interface.Variable(f"delta_{rxn}", type="binary") for rxn in rxns_g}
    
    # Variables duales para TODOS los metabolitos
    lambda_dual = {met: gurobi_interface.Variable(f"lambda_{met}", lb=-1000, ub=1000) 
                  for met in S_g_ng.index}
    
    # Variables de holgura complementaria
    mu_min = {rxn: gurobi_interface.Variable(f"mu_min_{rxn}", lb=0) for rxn in rxns_ng}
    mu_max = {rxn: gurobi_interface.Variable(f"mu_max_{rxn}", lb=0) for rxn in rxns_ng}
    z_min = {rxn: gurobi_interface.Variable(f"z_min_{rxn}", type="binary") for rxn in rxns_ng}
    z_max = {rxn: gurobi_interface.Variable(f"z_max_{rxn}", type="binary") for rxn in rxns_ng}
    
    # 5. Restricciones principales
    print("\nAdding constraints...")
    
    # Balance de masa para crecimiento
    for met in S_g_ng.index:
        expr = sum(S_g_ng.at[met, rxn] * v_g[rxn] for rxn in rxns_g 
                 if not np.isclose(S_g_ng.at[met, rxn], 0))
        opt_model.add(gurobi_interface.Constraint(expr, lb=0, ub=0, name=f"growth_{met}"))
    
    # Biomasa mínima
    opt_model.add(gurobi_interface.Constraint(
        v_g[biomass_rxn_id], lb=T_g, name="min_biomass"))
    
    # Restricción clave: ATP = 0 en microbioma
    if f"ng_{atp_rxn_id}" in rxns_ng:
        opt_model.add(gurobi_interface.Constraint(
            v_ng[f"ng_{atp_rxn_id}"], lb=0, ub=0, name="zero_atp_microbiome"))
    
    # 6. Condiciones KKT completas
    print("  - Adding KKT conditions...")
    M = 1e6  # Big-M
    
    # Para cada reacción no asociada a crecimiento:
    for rxn in rxns_ng:
        # Gradiente Lagrangiano
        grad_expr = sum(S_g_ng.at[met, rxn] * lambda_dual[met] 
                      for met in S_g_ng.index if not np.isclose(S_g_ng.at[met, rxn], 0))
        
        if rxn == f"ng_{atp_rxn_id}":
            grad_expr += -1  # Coeficiente en la función objetivo
            
        grad_expr += mu_min[rxn] - mu_max[rxn]
        opt_model.add(gurobi_interface.Constraint(
            grad_expr, lb=0, ub=0, name=f"lagrangian_{rxn}"))
        
        # Holgura complementaria (linealizada)
        rxn_orig = rxn.replace("ng_", "")
        lb = model.reactions.get_by_id(rxn_orig).lower_bound
        ub = model.reactions.get_by_id(rxn_orig).upper_bound
        
        # Para cotas inferiores
        opt_model.add(gurobi_interface.Constraint(
            v_ng[rxn] - lb * (1 - delta[rxn_orig]) <= M * (1 - z_min[rxn]),
            name=f"comp_min1_{rxn}"))
        opt_model.add(gurobi_interface.Constraint(
            mu_min[rxn] <= M * z_min[rxn],
            name=f"comp_min2_{rxn}"))
        
        # Para cotas superiores
        opt_model.add(gurobi_interface.Constraint(
            ub * (1 - delta[rxn_orig]) - v_ng[rxn] <= M * (1 - z_max[rxn]),
            name=f"comp_max1_{rxn}"))
        opt_model.add(gurobi_interface.Constraint(
            mu_max[rxn] <= M * z_max[rxn],
            name=f"comp_max2_{rxn}"))
    
    # 7. Objetivo: minimizar número de reacciones activas
    opt_model.objective = gurobi_interface.Objective(
        sum(delta.values()), direction="min")
    
    return opt_model

In [22]:
model_test0=deepcopy(model)

Read LP format model from file C:\Users\diazt\AppData\Local\Temp\tmpfav3n17t.lp
Reading time = 0.11 seconds
: 1805 rows, 5166 columns, 20366 nonzeros


In [23]:
modelo_gf=globalfit_with_print(model=model_test0, S_g_ng_original=S_g_ng, biomass_rxn_id="BIOMASS_Ec_iJO1366_core_53p95M", atp_rxn_id="ATPM", T_g=0.0001, extra_exclude=[])


=== STARTING GLOBALFIT ===

Adding constraints...
  - Condiciones de complementariedad... 12 restricciones agregadas
  - Restricción de ATP en no-crecimiento... OK

Solving with Gurobi...


In [24]:
modelo_gf.optimize()

'infeasible'